In [1]:
import pandas as pd
import json
import re
from pathlib import Path
from collections import Counter

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Define the folder containing the raw Adzuna JSON files
DATA_DIR = Path("../data/raw/adzuna")

# Raw job-posting files collected for the project
files = [
    "adzuna_software_engineer_2026-09-29.json",
    "adzuna_business_analyst_2026-09-29.json",
    "adzuna_data_analyst_2026-09-29.json",
    "adzuna_digital_marketing_2026-09-29.json"
]

print("Data directory:", DATA_DIR)
print("Files to load:")
for file in files:
    print("-", file)

Data directory: ../data/raw/adzuna
Files to load:
- adzuna_software_engineer_2026-09-29.json
- adzuna_business_analyst_2026-09-29.json
- adzuna_data_analyst_2026-09-29.json
- adzuna_digital_marketing_2026-09-29.json


In [3]:
# Load all job postings from the raw JSON files

all_jobs = []

for file in files:
    file_path = DATA_DIR / file
    
    with open(file_path, "r", encoding="utf-8") as f:
        jobs = json.load(f)
    
    print(f"{file}: {len(jobs)} records")
    all_jobs.extend(jobs)

print("\nTotal job postings loaded:", len(all_jobs))

adzuna_software_engineer_2026-09-29.json: 625 records
adzuna_business_analyst_2026-09-29.json: 625 records
adzuna_data_analyst_2026-09-29.json: 625 records
adzuna_digital_marketing_2026-09-29.json: 625 records

Total job postings loaded: 2500


In [4]:
# Convert the combined job postings into a DataFrame
df = pd.DataFrame(all_jobs)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (2500, 17)

Columns:
['redirect_url', 'description', 'created', 'contract_time', 'adref', 'id', 'location', 'company', 'longitude', 'latitude', '__CLASS__', 'category', 'salary_is_predicted', 'title', 'contract_type', 'salary_max', 'salary_min']


In [5]:
# Inspect the fields relevant to text analysis

print("Description data type:")
print(df["description"].map(type).value_counts())

print("\nMissing values:")
print(df[["id", "title", "description", "category"]].isnull().sum())

print("\nSample job description:")
print(df["description"].iloc[0][:1000])

Description data type:
description
<class 'str'>    2500
Name: count, dtype: int64

Missing values:
id             0
title          0
description    0
category       0
dtype: int64

Sample job description:
This job is with ABB, an inclusive employer and a member of myGwork – the largest global platform for the LGBTQ business community. Please do not contact the recruiter directly. At ABB, we help industries run leaner and cleaner—and every person here makes that happen. You’ll be empowered to lead, supported to grow, and proud of the impact we create together. Join us and help run what runs the world. This Position reports to: Digital Solution Engineering Manager What we believe in ABB’s Process …


In [6]:
# Extract fields required for text cleaning and EDA

text_df = df[["id", "title", "category", "description"]].copy()

# Extract the category label from the nested category field
text_df["category_name"] = text_df["category"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)

# Rename the job identifier for clarity
text_df.rename(columns={"id": "job_id"}, inplace=True)

# Keep only the fields needed for the text-data track
text_df = text_df[
    ["job_id", "title", "category_name", "description"]
]

print("Text dataset shape:", text_df.shape)
print("\nColumns:")
print(text_df.columns.tolist())

print("\nJob categories:")
print(text_df["category_name"].value_counts())

Text dataset shape: (2500, 4)

Columns:
['job_id', 'title', 'category_name', 'description']

Job categories:
category_name
IT Jobs                             1713
PR, Advertising & Marketing Jobs     494
Accounting & Finance Jobs             72
Sales Jobs                            70
Engineering Jobs                      43
Creative & Design Jobs                37
Consultancy Jobs                      16
HR & Recruitment Jobs                 12
Teaching Jobs                         10
Admin Jobs                             9
Energy, Oil & Gas Jobs                 7
Scientific & QA Jobs                   5
Logistics & Warehouse Jobs             3
Trade & Construction Jobs              2
Legal Jobs                             2
Graduate Jobs                          2
Manufacturing Jobs                     1
Healthcare & Nursing Jobs              1
Customer Services Jobs                 1
Name: count, dtype: int64


In [7]:
# Preserve the original job description before preprocessing

text_df["description_original"] = text_df["description"]

# Create a separate working column for text preprocessing
text_df["description_clean"] = text_df["description"]

print("Original description preserved:", 
      text_df["description_original"].notna().all())

print("Clean description column created:", 
      "description_clean" in text_df.columns)

print("\nCurrent shape:", text_df.shape)

Original description preserved: True
Clean description column created: True

Current shape: (2500, 6)
